# Lasso Regression

## 1. What is Lasso Regression?

Lasso = **Least Absolute Shrinkage and Selection Operator**.

It modifies Linear Regression by adding an **L1 regularization penalty**:

$$
Loss = MSE + \lambda \sum |w_j|
$$

Compare:

### Linear Regression

$$
Loss = MSE
$$

### Ridge

$$
Loss = MSE + \lambda\sum w_j^2
$$

### Lasso

$$
Loss = MSE + \lambda\sum |w_j|
$$

The important difference is:

| Model             | Penalty     | Can coefficient become exactly 0? |
| ----------------- | ----------- | --------------------------------- |
| Linear Regression | None        | ❌                                |
| Ridge             | \(L2: w^2\) | Usually ❌                        |   
| Lasso             | (L1:\|w\|) | ✅ |

That last column is the whole game.

### Why do we need Lasso?

- Imagine we have 10 features for predicting employee salary:

```text
experience
education
age
skills
projects
certifications
city
company_size
linkedin_connections
random_noise_feature
```

- Maybe only 5–6 actually matter. Without regularization, the model might assign some weight to almost everything. 

- Lasso can effectively do:
```text
experience             →  2.8
education              →  1.5
skills                 →  3.1
projects               →  0.8
certifications         →  0.2
city                   →  0.0
linkedin_connections   →  0.0
random_noise_feature   →  0.0
```
- Those zero coefficients mean: > The model has effectively removed those features.

- That's why Lasso is associated with feature selection.

- L1 regularization encourages sparse models.

### `alpha` in Lasso

Just like Ridge:

```python
from sklearn.linear_model import Lasso

model = Lasso(alpha=1.0)
```

`alpha` controls regularization strength.

### Small alpha

```text
alpha ↓
regularization ↓
coefficients less constrained
```

### Large alpha

```text
alpha ↑
regularization ↑
more coefficients pushed toward zero
```

If alpha becomes too large:

```text
too many coefficients → 0
             ↓
important information lost
             ↓
underfitting
```

So again, we're looking for a useful balance.

In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

np.random.seed(42)

X = np.random.randn(200, 10)

# Only first 3 features actually matter
y = (
    5 * X[:, 0]
    + 3 * X[:, 1]
    - 2 * X[:, 2]
    + np.random.randn(200) * 0.5
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# Train Lasso
lasso = make_pipeline(
    StandardScaler(),
    Lasso(alpha=0.1)
)

lasso.fit(X_train, y_train)

#predict
y_pred = lasso.predict(X_test)

# Evaluate the model
from sklearn.metrics import mean_squared_error

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("RMSE:", rmse)


#inspect the coefficients
lasso_model = lasso.named_steps["lasso"]

print(lasso_model.coef_)

RMSE: 0.5210000649274742
[ 4.56579762  3.28220451 -1.85301194  0.          0.         -0.
 -0.         -0.         -0.          0.        ]


In [3]:
# Compare Linear vs Ridge vs Lasso
linear = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

ridge = make_pipeline(
    StandardScaler(),
    Ridge(alpha=0.1)
)

lasso = make_pipeline(
    StandardScaler(),
    Lasso(alpha=0.1)
)

models = {
    "Linear": linear,
    "Ridge": ridge,
    "Lasso": lasso
}

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(name, "RMSE:", rmse)


# inspect the coefficients
print("Ridge coefficients:")
print(ridge.named_steps["ridge"].coef_)

print("\nLasso coefficients:")
print(lasso.named_steps["lasso"].coef_)

Linear RMSE: 0.4412239984871242
Ridge RMSE: 0.4416113892237693
Lasso RMSE: 0.5210000649274742
Ridge coefficients:
[ 4.70685386  3.38629279 -1.98953908  0.02699353  0.0286265  -0.04063871
 -0.03780199 -0.02768018 -0.02459136  0.06664823]

Lasso coefficients:
[ 4.56579762  3.28220451 -1.85301194  0.          0.         -0.
 -0.         -0.         -0.          0.        ]


# 9. Ridge vs Lasso — VERY IMPORTANT

This is a classic interview question.

### Ridge

$$
MSE + \lambda\sum w^2
$$

Main behavior:

```text
Coefficient
    ↓
smaller
    ↓
smaller
    ↓
usually not exactly zero
```

Useful when:

* many features contribute
* features are correlated
* you want coefficient shrinkage
* you don't necessarily want feature elimination

---

### Lasso

$$
MSE + \lambda\sum |w|
$$

Main behavior:

```text
Coefficient
    ↓
smaller
    ↓
smaller
    ↓
possibly ZERO
```

Useful when:

* you have many features
* some may be irrelevant
* you want sparse models
* feature selection is useful


In [6]:
# Lasso + Polynomial

from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Lasso

model = make_pipeline(
    PolynomialFeatures(degree=10),
    StandardScaler(),
    Lasso(alpha=0.01)
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(np.sqrt(mean_squared_error(y_test, y_pred)))

0.6858410258856282


In [8]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso

np.random.seed(42)

X = np.random.randn(300, 10)

y = (
    6 * X[:, 0]
    + 4 * X[:, 1]
    - 3 * X[:, 2]
    + np.random.randn(300)
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# testing
alphas = [0.001, 0.01, 0.1, 0.5, 1, 5]

for alpha in alphas:

    model = make_pipeline(
        StandardScaler(),
        Lasso(alpha=alpha)
    )

    model.fit(X_train, y_train)

    lasso_model = model.named_steps["lasso"]

    zero_count = np.sum(lasso_model.coef_ == 0)

    print(
        f"alpha={alpha:<5} "
        f"zero_coefficients={zero_count} "
        f"coefficients={np.round(lasso_model.coef_, 2)}"
    )

alpha=0.001 zero_coefficients=0 coefficients=[ 5.45  4.44 -3.01  0.08  0.06 -0.   -0.03 -0.04 -0.07 -0.03]
alpha=0.01  zero_coefficients=1 coefficients=[ 5.44  4.44 -3.    0.07  0.05 -0.   -0.02 -0.03 -0.06 -0.02]
alpha=0.1   zero_coefficients=7 coefficients=[ 5.35  4.36 -2.9   0.    0.   -0.   -0.   -0.   -0.   -0.  ]
alpha=0.5   zero_coefficients=7 coefficients=[ 4.94  3.99 -2.48  0.    0.   -0.    0.   -0.   -0.   -0.  ]
alpha=1     zero_coefficients=7 coefficients=[ 4.43  3.54 -1.94  0.    0.    0.    0.   -0.   -0.   -0.  ]
alpha=5     zero_coefficients=9 coefficients=[0.51 0.   0.   0.   0.   0.   0.   0.   0.   0.  ]
